## Rhetoric

##### 1 - Install DocuScope

Installs pandas, polars, pyarrow and docuscospacy, and the `en_docusco_spacy` model (version 1.5). pyarrow reads and writes the parquet files.

In [ ]:
%pip install pandas polars pyarrow docuscospacy
%pip install "https://huggingface.co/browndw/en_docusco_spacy/resolve/main/en_docusco_spacy-1.5-py3-none-any.whl"

##### 2 - Tag the pool with DocuScope

Loads the `en_docusco_spacy` model and tags every email in the pool (`in_pool`, from `corpus.ipynb` block 3). Saves every tag, with its category, phrase and character offsets, to `tags.parquet`, and the parsed emails to `docs.spacy`, so that later notebooks read the tags without parsing again. Counts are tag spans.

In [ ]:
import time
import pandas as pd
import spacy
from spacy.tokens import DocBin
from pathlib import Path

# Repository root (the folder with _quarto.yml), found from this notebook's folder
root = Path.cwd()
while not (root / '_quarto.yml').exists():
    root = root.parent
here = root / 'coding' / 'rhetoric'

nlp = spacy.load('en_docusco_spacy')

emails_study = pd.read_parquet(root / 'corpus' / 'emails_study.parquet')
pool = emails_study[emails_study['in_pool']]
print(f"Emails in pool: {len(pool):,}")

# Parse all emails in the pool, one row per tagged phrase
start = time.time()
docs = DocBin(store_user_data=True)
rows = []
for msgid2, doc in zip(pool['msgid2'], nlp.pipe(pool['body'].astype(str), batch_size=50)):
    doc.user_data['msgid2'] = str(msgid2)
    docs.add(doc)
    for ent in doc.ents:
        rows.append((msgid2, ent.label_, " ".join(ent.text.split()), ent.start_char, ent.end_char))
tags = pd.DataFrame(rows, columns=['msgid2', 'tag', 'phrase', 'start', 'end'])
print(f"Parsed in {(time.time() - start) / 60:.1f} min: {len(tags):,} tags")

tags.to_parquet(here / 'tags.parquet', index=False)
docs.to_disk(here / 'docs.spacy')
print("Saved to:", here / 'tags.parquet', "and", here / 'docs.spacy')

##### 3 - Distribution of HOW tags

Keeps only the HOW categories of the tags from block 2 and summarizes their distribution by category, email and sender. Counts are tag spans.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# The 19 HOW categories
how_cats = {"Interactive", "Facilitate", "MetadiscourseInteractive", "Inquiry",
            "ConfidenceHigh", "ConfidenceHedged", "ForceStressed", "Uncertainty", "Contingent", "Responsibility",
            "Positive", "Negative", "Future", "Updates", "Narrative", "Strategic",
            "FirstPerson", "MetadiscourseCohesive", "Reasoning"}

spans = tags[['msgid2', 'tag', 'phrase']]

# Keep HOW categories only
how = spans[spans['tag'].isin(how_cats)].copy()
print(f"HOW tags: {len(how):,} of {len(spans):,} ({100 * len(how) / len(spans):.1f}%)")

# HOW totals per category, as % of HOW tags and % of all tags
cat_tab = how['tag'].value_counts().rename('tags').to_frame()
cat_tab['pct_of_how'] = 100 * cat_tab['tags'] / len(how)
cat_tab['pct_of_all'] = 100 * cat_tab['tags'] / len(spans)
display(cat_tab.round(1))
cat_order = list(cat_tab.index)

# HOW tags per email, and HOW share of each email's tags (emails with no tags kept as 0)
per_email = pd.DataFrame({
    'all_tags': spans.groupby('msgid2').size(),
    'how_tags': how.groupby('msgid2').size()
}).reindex(pool['msgid2']).fillna(0)
per_email['how_share'] = per_email['how_tags'] / per_email['all_tags'].replace(0, np.nan)
print("Emails with no HOW tags:", int((per_email['how_tags'] == 0).sum()))
display(per_email[['how_tags', 'how_share']].describe().round(2))

fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
per_email['how_tags'].plot.hist(bins=40, ax=axes[0])
axes[0].set_xlabel('HOW tags per email')
per_email['how_share'].dropna().plot.hist(bins=40, ax=axes[1])
axes[1].set_xlabel('HOW share of all tags per email')
plt.tight_layout()
plt.show()

# Mix of categories within each email (proportions of the email's HOW tags)
cat_props = (how.groupby(['msgid2', 'tag']).size().unstack(fill_value=0)
             .reindex(columns=cat_order, fill_value=0))
cat_props = cat_props.div(cat_props.sum(axis=1), axis=0)
ax = cat_props.plot.box(figsize=(12, 4), rot=90)
ax.set_ylabel("Share of the email's HOW tags")
plt.show()
display(cat_props.describe().round(2))

# Mix of categories for the 10 most frequent senders
sender = pool.set_index('msgid2')['from']
sender_counts = sender.value_counts()
how['sender'] = how['msgid2'].map(sender)
by_sender = (how[how['sender'].isin(sender_counts.head(10).index)]
             .groupby(['sender', 'tag']).size().unstack(fill_value=0)
             .reindex(columns=cat_order, fill_value=0))
by_sender = by_sender.div(by_sender.sum(axis=1), axis=0)
by_sender.insert(0, 'emails', sender_counts.reindex(by_sender.index))
display(by_sender.sort_values('emails', ascending=False).round(2))

# 15 most frequent phrases per HOW category (lowercased)
how['phrase_lc'] = how['phrase'].str.lower()
top_phrases = (how.groupby(['tag', 'phrase_lc']).size().rename('n').reset_index()
               .sort_values(['tag', 'n'], ascending=[True, False])
               .groupby('tag').head(15))
for cat in cat_order:
    sub = top_phrases[top_phrases['tag'] == cat]
    print(f"\n{cat} ({int((how['tag'] == cat).sum()):,} tags)")
    print("  " + "; ".join(f"{p} ({n})" for p, n in zip(sub['phrase_lc'], sub['n'])))